# 01 · Load Lending Club data & column triage

**Session 1 goal:** get the accepted-loans dataset into SQLite, confirm its
shape, and produce an *annotated data dictionary* so you can do the
keep / drop / **leakage** triage that drives the whole project.

The golden rule for every column: **would a lender know this the day the
application arrives?** If not, it is leakage and must be excluded.

Pipeline: Kaggle `.csv.gz` → `data/raw/` → SQLite table `loans` → data dictionary.


In [ ]:
import sqlite3
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Project root = parent of the notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

RAW_CSV = ROOT / "data" / "raw" / "accepted_2007_to_2018Q4.csv.gz"
DB_PATH = ROOT / "data" / "lending_club.db"
REPORTS = ROOT / "reports"
print("ROOT      :", ROOT)
print("raw csv   :", RAW_CSV, "->", "present" if RAW_CSV.exists() else "MISSING")
print("sqlite db :", DB_PATH, "->", "present" if DB_PATH.exists() else "MISSING")

## Step 1 — Download the dataset

Needs a Kaggle API token. Two ways to set one up (one-time):

- **OAuth (easiest):** `./venv/bin/kaggle auth login` and follow the browser prompt, **or**
- **Token file:** create one at <https://www.kaggle.com/settings> → *API* → *Create New Token*,
  then `mkdir -p ~/.kaggle && mv ~/Downloads/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json`

The cell below downloads the file only if it isn't already in `data/raw/`.
If you downloaded the CSV manually, just drop it in `data/raw/` (keep the name
`accepted_2007_to_2018Q4.csv.gz`) and skip this cell.

In [ ]:
if RAW_CSV.exists():
    print("Already downloaded — skipping.")
else:
    print("Downloading via src/download_data.py ...")
    result = subprocess.run(
        [sys.executable, str(SRC / "download_data.py")],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        print("\nDownload failed — see the message above (usually missing Kaggle token).")

## Step 2 — Load into SQLite (single table `loans`)

The file is ~2.2M rows × 151 cols, so the loader streams it in chunks and
appends, keeping memory bounded. It rebuilds the DB from scratch each run so
it's idempotent.

In [ ]:
if DB_PATH.exists():
    print("DB already built — skipping. Delete data/lending_club.db to rebuild.")
elif RAW_CSV.exists():
    result = subprocess.run(
        [sys.executable, str(SRC / "load_to_sqlite.py")],
        capture_output=True, text=True,
    )
    print(result.stdout[-500:])
    if result.returncode != 0:
        print(result.stderr)
else:
    print("No CSV yet — complete Step 1 first.")

## Step 3 — Shape & the target distribution

In [ ]:
con = sqlite3.connect(DB_PATH)

n_rows = pd.read_sql("SELECT COUNT(*) AS n FROM loans", con).iloc[0, 0]
cols = pd.read_sql("SELECT * FROM loans LIMIT 1", con).columns.tolist()
print(f"Shape: {n_rows:,} rows x {len(cols)} columns\n")

# The target lives in loan_status. Here's the raw distribution.
status = pd.read_sql(
    "SELECT loan_status, COUNT(*) AS n FROM loans GROUP BY loan_status ORDER BY n DESC",
    con,
)
status["pct"] = (100 * status["n"] / status["n"].sum()).round(2)
print("loan_status distribution:")
print(status.to_string(index=False))

**Reading this:** `Fully Paid` → label 0, `Charged Off` / `Default` → label 1.
The `Current`, `Late`, `In Grace Period`, and `Issued` loans have no final
outcome yet — the clean Session-1 move is to drop them so every row has a
known result. We'll formalize the target in Session 2.

## Step 4 — Annotated data dictionary

For every column we compute empirical stats (dtype, null %, cardinality,
sample values) and attach a **leakage flag** + description from
`src/lc_data_dictionary.py`. This table is your triage worksheet.

In [ ]:
import lc_data_dictionary as lcd

# Missing % on the FULL 2.2M rows (via SQL). The file is roughly
# chronological and LC added columns over the years, so a naive
# `LIMIT 50000` would grab the oldest loans and falsely report newer
# columns (joint_*, hardship_*, sec_app_*) as ~100% empty. Compute it
# unbiased across all rows.
null_exprs = ", ".join(
    f'SUM(CASE WHEN "{c}" IS NULL OR "{c}"=\'\' THEN 1 ELSE 0 END) AS "{c}"'
    for c in cols
)
null_counts = pd.read_sql(f"SELECT {null_exprs} FROM loans", con).iloc[0]
pct_null_full = (100 * null_counts / n_rows).round(2)

# Random sample only for dtype / cardinality / example values.
sample = pd.read_sql(
    "SELECT * FROM loans WHERE rowid IN "
    "(SELECT rowid FROM loans ORDER BY RANDOM() LIMIT 60000)", con
)

rows = []
for col in cols:
    s = sample[col]
    examples = s.dropna().astype(str).str.slice(0, 30).unique()[:3].tolist()
    flag, desc = lcd.annotate(col)
    rows.append({
        "column": col,
        "leakage_flag": flag,
        "dtype": str(s.dtype),
        "pct_null_full": float(pct_null_full[col]),   # over all 2.2M rows
        "n_unique_sample": int(s.nunique(dropna=True)),
        "examples": " | ".join(examples),
        "description": desc,
        "keep_decision": "",   # <- you fill this in during triage
    })

data_dict = pd.DataFrame(rows)

# Order so the decisions that matter jump out: leakage & target first.
flag_order = {"TARGET": 0, "LEAKAGE": 1, "LC_MODEL": 2, "REVIEW": 3, "SAFE": 4}
data_dict = data_dict.sort_values(
    by=["leakage_flag", "column"],
    key=lambda s: s.map(flag_order).fillna(9) if s.name == "leakage_flag" else s,
).reset_index(drop=True)

print("Flag summary:")
print(data_dict["leakage_flag"].value_counts().to_string())

In [ ]:
# Save the dictionary for the README / triage worksheet.
REPORTS.mkdir(exist_ok=True)
data_dict.to_csv(REPORTS / "data_dictionary.csv", index=False)

# Also a Markdown version to paste into the README.
with open(REPORTS / "data_dictionary.md", "w") as f:
    f.write("# Lending Club — data dictionary & leakage triage\n\n")
    f.write(f"{n_rows:,} rows x {len(cols)} columns.\n\n")
    f.write("Flags: TARGET / SAFE (use) / LC_MODEL (use deliberately) / "
            "LEAKAGE (exclude) / REVIEW (decide).\n\n")
    f.write(data_dict.to_markdown(index=False))

print("Wrote reports/data_dictionary.csv and reports/data_dictionary.md")
data_dict

## What to do next (your triage)

1. Open `reports/data_dictionary.csv` (or the rendered table above).
2. For each row, fill `keep_decision` with **keep / drop / review-later**.
   - Everything flagged `LEAKAGE` → **drop**.
   - `LC_MODEL` (grade/sub_grade/int_rate) → decide: build one model with,
     one without, to see how much signal is just LC's own grade.
   - High-null or high-cardinality free-text (`emp_title`, `title`, `desc`,
     `url`) → usually drop or needs feature engineering.
3. Record the final keep-list and your reasoning in the README.

Then Session 2: define the target, split train/test, and fit a logistic
baseline.